In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable


In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "gross_price", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source}')

display(df_silver)

In [0]:
df_gold = df_silver.select("product_code", "month", "gross_price")

df_gold.show(10)

In [0]:
df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed', 'True')\
    .option('mergeSchema', 'True')\
    .mode('overwrite')\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [0]:
df_gold_price = spark.table(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

display(df_gold_price)

In [0]:
df_gold_price = (
                df_gold_price
                 .withColumn('year', F.year('month'))
                .withColumn('is_zero', F.when(F.col('gross_price') == 0 , 1)
                .otherwise(0))
            )

w = (
    Window
    .partitionBy('product_code', 'year')
    .orderBy(F.col('is_zero'), F.col('month').desc())
)

df_gold_latest_price = (
    df_gold_price
    .withColumn('rank', F.row_number().over(w))
    .filter(F.col('rank') == 1)
)

In [0]:
display(df_gold_latest_price)

In [0]:
df_gold_latest_price = df_gold_latest_price.select("product_code", "year", "gross_price").withColumnRenamed("gross_price", "price_inr").select("product_code", "price_inr", "year")

df_gold_latest_price = df_gold_latest_price.withColumn("year", F.col("year").cast("string"))

df_gold_latest_price.show(5)

In [0]:
df_gold_latest_price.printSchema()

In [0]:
df_parent_pricing = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.dim_{data_source}")


df_parent_pricing.alias("target").merge(
    source=df_gold_latest_price.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()